**Tumour-Only WT Matching for Individual-Gene Cohorts**
This notebook creates sex- and cancer-matched WT controls for each individual-gene/cohort analysis.

Controls are restricted to primary tumour samples (sample_type_code == "01") -> avoids accidentally using adjacent-normal, metastatic, or recurrent samples as WT controls.

The key output for each cohort is:
<gene_set_label>_case_control_matched_table_ca

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
notebook_dir <- file.path(project_root, "23_individualgenes_WTmatching_tumouronly")
dir.create(notebook_dir, showWarnings = FALSE, recursive = TRUE)
setwd(notebook_dir)

tedata_match_input_dir <- "../22_individualgenes_TEdata_overlap"

# Extracted REdiscoverTE data folder
tedata_root <- "../TEdata_resources"

match_output_dir <- "."

# Two controls per altered case -> matching your existing analysis
n_controls_per_case <- 2

set.seed(1)

dir.exists(tedata_match_input_dir)
dir.exists(tedata_root)
dir.exists(match_output_dir)

In [ ]:
# Define cohorts to process

analysis_definitions <- tibble::tribble(
  ~gene_set_label, ~gene_set_group, ~display_name, ~genes,
  "kdm6a_loss",   "histone_modifier", "KDM6A loss",   list(c("KDM6A")),
  "kdm6b_loss",   "histone_modifier", "KDM6B loss",   list(c("KDM6B")),
  "nsd1_loss",    "histone_modifier", "NSD1 loss",    list(c("NSD1")),
  "nsd2_loss",    "histone_modifier", "NSD2 loss",    list(c("NSD2")),
  "setd2_loss",   "histone_modifier", "SETD2 loss",   list(c("SETD2")),
  "kdm6a_b_loss", "histone_modifier", "KDM6A/B loss", list(c("KDM6A", "KDM6B")),
  "nsd1_2_loss",  "histone_modifier", "NSD1/2 loss",  list(c("NSD1", "NSD2")),
  "arid1a_loss",   "swi_snf", "ARID1A loss",   list(c("ARID1A")),
  "arid1b_loss",   "swi_snf", "ARID1B loss",   list(c("ARID1B")),
  "arid1a_b_loss", "swi_snf", "ARID1A/B loss", list(c("ARID1A", "ARID1B")),
  "smarca4_loss",  "swi_snf", "SMARCA4 loss",  list(c("SMARCA4")),
  "smarcb1_loss", "swi_snf", "SMARCB1 loss", list(c("SMARCB1")),
  "pbrm1_loss",   "swi_snf", "PBRM1 loss",   list(c("PBRM1"))
)

In [ ]:
# Load REdiscoverTE metadata

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(rds_hits) == 1)
dat <- readRDS(rds_hits[1])

pdat <- pData(dat) %>%
  tibble::rownames_to_column("TEdata_full_id")

tedata_annot <- tibble(
  TEdata_full_id = sampleNames(dat),
  sample_id_16 = substr(sampleNames(dat), 1, 16),
  patient_id = substr(sampleNames(dat), 1, 12),
  sample_type_code = substr(sampleNames(dat), 14, 15)
) %>%
  left_join(pdat, by = "TEdata_full_id")

# REdiscoverTE metadata columns used for cancer and sex matching
project_col <- "indication"
sex_col <- "Sex"

stopifnot(project_col %in% colnames(tedata_annot))
stopifnot(sex_col %in% colnames(tedata_annot))

tedata_match <- tedata_annot %>%
  mutate(
    match_project = as.character(.data[[project_col]]),
    match_sex = as.character(.data[[sex_col]]),
    match_project = na_if(match_project, ""),
    match_sex = na_if(match_sex, "")
  )

# Check of available TEdata sample types
tedata_match %>% count(sample_type_code, sort = TRUE)

In [ ]:
# Set matching helper functions

match_one_case <- function(case_row, wt_df, n_controls = 2) {
  # Match first on tumour type/cancer indication
  candidates <- wt_df %>%
    filter(match_project == case_row$match_project)

  # Then match on sex where sex is available
  if (!is.na(case_row$match_sex) && case_row$match_sex != "") {
    candidates <- candidates %>%
      filter(match_sex == case_row$match_sex)
  }

  if (nrow(candidates) == 0) {
    return(tibble())
  }

  candidates %>%
    slice_sample(n = min(n_controls, nrow(candidates))) %>%
    mutate(
      case_TEdata_full_id = case_row$TEdata_full_id,
      case_sample_id_16 = case_row$sample_id_16,
      case_patient_id = case_row$patient_id,
      case_sample_type_code = case_row$case_sample_type_code,
      case_project = case_row$match_project,
      case_sex = case_row$match_sex,
      gene_set = case_row$gene_set,
      gene_set_group = case_row$gene_set_group,
      gene_set_display = case_row$gene_set_display,
      genes_tested = case_row$genes_tested,
      genes_lost = case_row$genes_lost,
      events = case_row$events,
      lof_genes = case_row$lof_genes,
      homdel_genes = case_row$homdel_genes,
      mutation_classes = case_row$mutation_classes,
      n_genes_lost = case_row$n_genes_lost,
      n_loss_events = case_row$n_loss_events,
      has_lof = case_row$has_lof,
      has_homdel = case_row$has_homdel
    )
}

match_one_cohort <- function(def_row) {
  gene_set_label <- def_row$gene_set_label
  message("===== ", gene_set_label, " =====")

  case_file <- file.path(
    tedata_match_input_dir,
    paste0("pancancer_", gene_set_label, "_samples_with_RNA_match_TE_ready.csv")
  )

  if (!file.exists(case_file)) {
    warning("Missing TE-ready case file for ", gene_set_label, ": ", case_file)
    return(tibble(
      gene_set = gene_set_label,
      gene_set_display = def_row$display_name,
      genes_tested = paste(unique(unlist(def_row$genes, use.names = FALSE)), collapse = ";"),
      n_cases = 0,
      n_controls = 0,
      n_rows = 0,
      n_cases_without_controls = NA_integer_,
      status = "missing_TE_ready_file"
    ))
  }

  loss_cases <- read_csv(case_file, show_col_types = FALSE) %>%
    mutate(
      sample_id_16 = substr(as.character(sample_id_16), 1, 16),
      patient_id = substr(as.character(patient_id), 1, 12),
      case_sample_type_code = substr(sample_id_16, 14, 15)
    ) %>%
    distinct()

  # Keep primary tumour altered cases only
  loss_cases_primary <- loss_cases %>%
    filter(case_sample_type_code == "01")

  if (nrow(loss_cases_primary) == 0) {
    warning("No primary tumour altered cases for ", gene_set_label)
    return(tibble(
      gene_set = gene_set_label,
      gene_set_display = def_row$display_name,
      genes_tested = paste(unique(unlist(def_row$genes, use.names = FALSE)), collapse = ";"),
      n_cases = 0,
      n_controls = 0,
      n_rows = 0,
      n_cases_without_controls = NA_integer_,
      status = "no_primary_tumour_cases"
    ))
  }

  case_metadata <- loss_cases_primary %>%
    select(-any_of(c("TEdata_full_id", "match_project", "match_sex"))) %>%
    left_join(
      tedata_match %>% select(TEdata_full_id, sample_id_16, match_project, match_sex),
      by = "sample_id_16"
    ) %>%
    filter(!is.na(TEdata_full_id)) %>%
    distinct(sample_id_16, .keep_all = TRUE)

  case_sample_ids <- case_metadata$sample_id_16
  case_patient_ids <- case_metadata$patient_id

  wt_pool <- tedata_match %>%
    filter(sample_type_code == "01") %>%
    filter(!sample_id_16 %in% case_sample_ids) %>%
    filter(!patient_id %in% case_patient_ids) %>%
    select(
      TEdata_full_id,
      sample_id_16,
      patient_id,
      WT_sample_type_code = sample_type_code,
      match_project,
      match_sex
    ) %>%
    distinct()

  # Match cases one at a time, removing selected WT controls from the pool.
  case_rows <- split(case_metadata, seq_len(nrow(case_metadata)))
  available_wt_pool <- wt_pool
  matched_wt_list <- vector("list", length(case_rows))

  for (i in seq_along(case_rows)) {
    matched_i <- match_one_case(
      case_row = case_rows[[i]],
      wt_df = available_wt_pool,
      n_controls = n_controls_per_case
    )

    matched_wt_list[[i]] <- matched_i

    if (nrow(matched_i) > 0) {
      available_wt_pool <- available_wt_pool %>%
        filter(!sample_id_16 %in% matched_i$sample_id_16)
    }
  }

  matched_wt <- bind_rows(matched_wt_list)

  match_counts <- matched_wt %>%
    count(case_sample_id_16, name = "n_controls_found")

  case_match_summary <- case_metadata %>%
    left_join(match_counts, by = c("sample_id_16" = "case_sample_id_16")) %>%
    mutate(
      n_controls_found = ifelse(is.na(n_controls_found), 0L, n_controls_found),
      matched_successfully = n_controls_found > 0
    )

  case_control_tbl <- matched_wt %>%
    select(
      case_TEdata_full_id,
      case_sample_id_16,
      case_patient_id,
      case_sample_type_code,
      case_project,
      case_sex,
      gene_set,
      gene_set_group,
      gene_set_display,
      genes_tested,
      genes_lost,
      events,
      lof_genes,
      homdel_genes,
      mutation_classes,
      n_genes_lost,
      n_loss_events,
      has_lof,
      has_homdel,
      WT_TEdata_full_id = TEdata_full_id,
      WT_sample_id_16 = sample_id_16,
      WT_patient_id = patient_id,
      WT_sample_type_code,
      WT_project = match_project,
      WT_sex = match_sex
    )

  write_csv(case_match_summary, file.path(match_output_dir, paste0(gene_set_label, "_case_match_summary_cancer_sex_primary_tumour_only.csv")))
  write_csv(matched_wt, file.path(match_output_dir, paste0(gene_set_label, "_matched_WT_controls_cancer_sex_primary_tumour_only_raw.csv")))
  write_csv(case_control_tbl, file.path(match_output_dir, paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv")))

  cat("Primary tumour cases:", n_distinct(case_control_tbl$case_sample_id_16), "\n")
  cat("WT controls:", n_distinct(case_control_tbl$WT_sample_id_16), "\n")

  tibble(
    gene_set = gene_set_label,
    gene_set_display = def_row$display_name,
    genes_tested = paste(unique(unlist(def_row$genes, use.names = FALSE)), collapse = ";"),
    n_cases = n_distinct(case_control_tbl$case_sample_id_16),
    n_controls = n_distinct(case_control_tbl$WT_sample_id_16),
    n_rows = nrow(case_control_tbl),
    n_cases_without_controls = sum(!case_match_summary$matched_successfully),
    status = "matched"
  )
}

In [ ]:
# Run WT matching for all cohorts

definition_rows <- split(analysis_definitions, seq_len(nrow(analysis_definitions)))

wt_matching_summary <- purrr::map_dfr(definition_rows, match_one_cohort)

write_csv(
  wt_matching_summary,
  file.path(match_output_dir, "individual_gene_WT_matching_summary.csv")
)

wt_matching_summary %>% arrange(desc(n_cases))

In [ ]:
# Check that every expected case-control file was created

# Useful before moving to TE differential expression analysis
expected_case_control_files <- analysis_definitions %>%
  mutate(
    expected_file = file.path(
      match_output_dir,
      paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv")
    ),
    file_created = file.exists(expected_file)
  ) %>%
  select(gene_set_label, display_name, expected_file, file_created)

expected_case_control_files

# Show only missing files, if any
expected_case_control_files %>%
  filter(!file_created)